## 确定round内最后一个epoch的准则

### 构建伪验证集

In [11]:
import os, json
tmp_dir = '/data/home/scv7387/run/tv_series_plus/3D-Speaker/docs/可视化/temp_face'
audio_smoothed_json_path = os.path.join(tmp_dir, 'pseudo_labels_audio_nested_hmm_full(unreliable_pp=5.0).json')
face_cluster_aligned_json_path = os.path.join(tmp_dir, 'cluster_results_faces_mid_frame_processed_all_for_HMM_nested_X.json')

with open(audio_smoothed_json_path, 'r') as f:
    audio_smoothed_data = json.load(f)
with open(face_cluster_aligned_json_path, 'r') as f:
    face_cluster_aligned_data = json.load(f)

# 将face ids按audio_seg_id分组
face_ids_dic = {}
for face_id in face_cluster_aligned_data:
    audio_seg_id = face_id.rsplit('_', 1)[0]
    if audio_seg_id not in face_ids_dic:
        face_ids_dic[audio_seg_id] = []
    face_ids_dic[audio_seg_id].append(face_id)

# 根据共现信息筛选face ids(效果不佳)
face_ids_filtered_list_cooccur = []
for key in face_ids_dic:
    labels = [face_cluster_aligned_data[face_id] for face_id in face_ids_dic[key]]
    labels_filtered = [label for label in labels if label != -1]
    if len(labels_filtered) == len(set(labels_filtered)):
        face_ids_filtered_list_cooccur.extend(face_ids_dic[key])


face_ids_filtered_list_spk = []
for key in face_ids_dic:
    face_ids_spk = []
    for face_id in face_ids_dic[key]:
        if audio_smoothed_data[key] == face_cluster_aligned_data[face_id]:
            face_ids_spk.append(face_id)
    if len(face_ids_spk) == 1:
        face_ids_filtered_list_spk.extend(face_ids_spk)
# for key in face_ids_dic:
#     if len(face_ids_dic[key])==1:
#         if audio_smoothed_data[key] == face_cluster_aligned_data[face_ids_dic[key][0]]:
#             face_ids_filtered_list_spk.append(face_ids_dic[key][0])

print("Total number of face samples:", len(face_cluster_aligned_data))
print(f"Number of selected samples for pseudo valid set of face(according to co-occurrence): {len(face_ids_filtered_list_cooccur)}")
print(f"Number of selected samples for pseudo valid set of face(according to speaker): {len(face_ids_filtered_list_spk)}")
with open(os.path.join(tmp_dir, 'selected_keys_for_mid_frame_faces_cooccur.json'), 'w', encoding='utf-8') as f:
    json.dump({k: face_cluster_aligned_data[k] for k in face_ids_filtered_list_cooccur}, f, indent=2)
with open(os.path.join(tmp_dir, 'selected_keys_for_mid_frame_faces_spk.json'), 'w', encoding='utf-8') as f:
    json.dump({k: face_cluster_aligned_data[k] for k in face_ids_filtered_list_spk}, f, indent=2)

Total number of face samples: 14646
Number of selected samples for pseudo valid set of face(according to co-occurrence): 14387
Number of selected samples for pseudo valid set of face(according to speaker): 5156


### 评估伪验证集的选择能力

In [12]:
import numpy as np
def calculate_acc_fromdic(pred_dic, ref_dic):
    correct_num = sum(list(map(lambda k: 1 if pred_dic[k] == ref_dic[k] else 0, ref_dic.keys())))
    ref_total = len(ref_dic)
    return correct_num / ref_total if ref_total > 0 else 0

def calculate_acc_fromjson(pred_json_path, ref_json_path=os.path.join(tmp_dir, 'selected_keys_for_mid_frame_faces_spk.json')):
    with open(pred_json_path, 'r', encoding='utf-8') as f:
        pred_dic = json.load(f)
    with open(ref_json_path, 'r', encoding='utf-8') as f:
        ref_dic = json.load(f)
    return calculate_acc_fromdic(pred_dic, ref_dic)

def get_json_paths_ft(round_dir):
    ft_dirs = [d for d in os.listdir(round_dir) if os.path.isdir(os.path.join(round_dir, d)) and d.startswith("ft_epoch_face")]
    json_paths_ft = [os.path.join(round_dir, d, 'pseudo_labels_faces_mid_frame_pred.json') for d in ft_dirs]
    return json_paths_ft


exp_dir = '/data/home/scv7387/run/tv_series_plus/3D-Speaker/egs/3dspeaker/speaker-diarization/runs/the big bang theory/exp_video/result/self_supervised/exp1'
# 检查ref_json_path是否能用于确定 best epoch
round_dirs = [d for d in os.listdir(exp_dir) if os.path.isdir(os.path.join(exp_dir, d)) and d.startswith("round")]
round_dirs.sort()
for round_dir in round_dirs:
    json_paths = get_json_paths_ft(os.path.join(exp_dir, round_dir))
    json_paths.sort(key=lambda x: int(os.path.basename(os.path.dirname(x)).rsplit('_', 1)[-1]))
    epochs_names = [os.path.basename(os.path.dirname(p)) for p in json_paths]
    accs = np.array([calculate_acc_fromjson(p) for p in json_paths])
    for i in range(len(epochs_names)):
        print(f"Round {round_dir}, {epochs_names[i]} acc: {accs[i]}")
    best_epoch = epochs_names[np.argmax(accs)]
    print(f"Round {round_dir}, Best epoch: {best_epoch}, best acc: {max(accs)}")



Round round0, ft_epoch_face_0 acc: 0.9829325058184639
Round round0, ft_epoch_face_1 acc: 0.9924359968968193
Round round0, ft_epoch_face_2 acc: 0.9908844065166796
Round round0, ft_epoch_face_3 acc: 0.9937936384794415
Round round0, ft_epoch_face_4 acc: 0.9967028704422033
Round round0, ft_epoch_face_5 acc: 0.995733126454616
Round round0, ft_epoch_face_6 acc: 0.9982544608223429
Round round0, ft_epoch_face_7 acc: 0.9953452288595811
Round round0, ft_epoch_face_8 acc: 0.9963149728471683
Round round0, ft_epoch_face_9 acc: 0.9984484096198604
Round round0, ft_epoch_face_10 acc: 0.9980605120248255
Round round0, ft_epoch_face_11 acc: 0.9988363072148952
Round round0, ft_epoch_face_12 acc: 0.9980605120248255
Round round0, Best epoch: ft_epoch_face_11, best acc: 0.9988363072148952
Round round1, ft_epoch_face_0 acc: 0.9951512800620637
Round round1, ft_epoch_face_1 acc: 0.9955391776570985
Round round1, ft_epoch_face_2 acc: 0.997478665632273
Round round1, ft_epoch_face_3 acc: 0.9967028704422033
Round ro

### 完善人脸聚类与说话人的对齐

In [16]:
import os, json
import numpy as np
from statistics import median
from collections import Counter
TV_name = 'I love my family' # 'I love my family', 'the big bang theory'
result_dir = os.path.join('/data/home/scv7387/run/tv_series_plus/3D-Speaker/egs/3dspeaker/speaker-diarization/runs', TV_name, 'exp_video', 'result', 'self_supervised', 'initial_old', 'pseudo_label')
cluster_results_faces_mf_all_path = os.path.join(result_dir, 'cluster_results_faces_mid_frame_processed_all_for_HMM_nested_X.json')
cluster_results_faces_mf_aligned_path = os.path.join(result_dir, 'cluster_results_faces_mid_frame_vision-audio_aligned.json')
cluster_results_audio_processed_path = os.path.join(result_dir, 'cluster_results_audio_processed_for_HMM_nested_X.json')

with open(cluster_results_faces_mf_all_path, 'r') as f:
    vlabels_mf_processed_all = json.load(f)
with open(cluster_results_faces_mf_aligned_path, 'r') as f:
    vlabels_mf_aligned = json.load(f)
with open(cluster_results_audio_processed_path, 'r') as f:
    alabels_processed = json.load(f)

# 将所有key-->cluster id的字典转化为cluster id-->key列表的字典
def invert_dic(label_dic):
    cluster_ids = list(set(label_dic.values()))
    cluster_dic = {cluster_id: [] for cluster_id in cluster_ids}
    for key in label_dic:
        cluster_id = label_dic[key]
        cluster_dic[cluster_id].append(key)
    return cluster_dic

# 找到所有cluster id 未对齐的face id，并汇总该部分数据不同cluster id的数量
unaligned_face_ids = [face_id for face_id in vlabels_mf_processed_all if face_id not in vlabels_mf_aligned]
vlabels_mf_unaligned = {face_id: vlabels_mf_processed_all[face_id] for face_id in unaligned_face_ids}
unaligned_face_cluster_dic = invert_dic(vlabels_mf_unaligned)
print("Number of unaligned face cluster ids:", len(unaligned_face_cluster_dic))
for cluster_id in unaligned_face_cluster_dic:
    print(f"Cluster id {cluster_id}: {len(unaligned_face_cluster_dic[cluster_id])} face samples")

# 将alabels_processed整理为cluster id-->key列表的字典
audio_cluster_dic = invert_dic(alabels_processed)
print("Number of audio cluster ids:", len(audio_cluster_dic))
for cluster_id in audio_cluster_dic:
    print(f"Cluster id {cluster_id}: {len(audio_cluster_dic[cluster_id])} audio samples")
audio_cluster_sizes = [len(audio_cluster_dic[cid]) for cid in audio_cluster_dic]
# 选取未对齐人脸聚类中样本数大于等于音频聚类中位数的聚类id
major_unaligned_face_clusters = [cluster_id for cluster_id in unaligned_face_cluster_dic if len(unaligned_face_cluster_dic[cluster_id]) >= median(audio_cluster_sizes)]
print(f"Major unaligned face cluster ids (size >= {median(audio_cluster_sizes)}):", major_unaligned_face_clusters)

for face_cluster_id in unaligned_face_cluster_dic:
    face_ids = unaligned_face_cluster_dic[face_cluster_id]
    # 将face ids按audio_seg_id分组
    face_ids_dic = {}
    for face_id in face_ids:
        audio_seg_id = face_id.rsplit('_', 1)[0]
        if audio_seg_id not in face_ids_dic:
            face_ids_dic[audio_seg_id] = []
        face_ids_dic[audio_seg_id].append(face_id)
    # 筛选只含单一face id的audio_seg_id
    audio_seg_ids_filtered_spk = []
    for key in face_ids_dic:
        if len(face_ids_dic[key]) == 1:
            audio_seg_ids_filtered_spk.append(key)
    alabels_filtered_spk = [alabels_processed[k] for k in audio_seg_ids_filtered_spk]
    # 汇总alabels_filtered_spk中各个取值的数量
    alabels_filtered_count = Counter(alabels_filtered_spk)
    # 创建一个字典保存audio label到ratio的映射
    label_ratio_dict = {label: count / len(audio_cluster_dic[label]) for label, count in alabels_filtered_count.items()}

    # 使用四分位极差法判断outliers
    ratios = list(label_ratio_dict.values())
    q1 = np.percentile(ratios, 25)
    q3 = np.percentile(ratios, 75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers_upper = {label: ratio for label, ratio in label_ratio_dict.items() if ratio > upper_bound}
    if len(outliers_upper) == 1 and list(outliers_upper.values())[0] > 0.5:
        print(f"\nFor face_cluster_id {face_cluster_id}, Outlier audio labels based on ratio:")
        for label, ratio in outliers_upper.items():
            print(f"Audio label {label}: ratio {ratio:.4f}")
        print("Corresponding counts in filtered samples:")
        for label, count in alabels_filtered_count.items():
            print(f"Audio label {label}: {count} samples, ratio in cluster: {label_ratio_dict[label]:.4f}")

Number of unaligned face cluster ids: 12
Cluster id 12: 8284 face samples
Cluster id 13: 375 face samples
Cluster id 14: 242 face samples
Cluster id 15: 227 face samples
Cluster id 16: 170 face samples
Cluster id 17: 103 face samples
Cluster id 18: 53 face samples
Cluster id 19: 29 face samples
Cluster id 20: 27 face samples
Cluster id 21: 22 face samples
Cluster id 22: 16 face samples
Cluster id 23: 1 face samples
Number of audio cluster ids: 13
Cluster id 0: 3953 audio samples
Cluster id 1: 3817 audio samples
Cluster id 2: 3076 audio samples
Cluster id 3: 991 audio samples
Cluster id 4: 829 audio samples
Cluster id 5: 700 audio samples
Cluster id 6: 645 audio samples
Cluster id 7: 613 audio samples
Cluster id 8: 583 audio samples
Cluster id 9: 521 audio samples
Cluster id 10: 472 audio samples
Cluster id 11: 428 audio samples
Cluster id -1: 2597 audio samples
Major unaligned face cluster ids (size >= 700): [12]
